In [45]:
import pandas as pd
import numpy as np


In [46]:
email_main= pd.read_excel('../20feb.xlsx', sheet_name='email_main')


In [47]:
bill = pd.read_excel('../20feb.xlsx', sheet_name='bill')

In [48]:
email_main[['Level']].value_counts()

Level  
UG         185
PG         117
DIPLOMA      8
Name: count, dtype: int64

In [49]:
email_main['email id'].nunique()

159

In [50]:
bill[['Level']].value_counts()

Level  
UG         185
PG         117
DIPLOMA      8
Name: count, dtype: int64

In [51]:
bill['print'].unique()

array(['rcvd', 'zcalled on 17 and 23 jan, no answer',
       'zcalled on 17 jan, will send soon',
       'zcalled on 23 jan again, will send', 'zno answer'], dtype=object)

In [52]:
bill_rcvd = bill[bill['print'] == 'rcvd']

In [53]:
bill_rcvd["Level"].value_counts()

Level
UG         177
PG         113
DIPLOMA      8
Name: count, dtype: int64

In [54]:
bill_rcvd.info()

<class 'pandas.core.frame.DataFrame'>
Index: 298 entries, 0 to 297
Data columns (total 21 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Sr. No.                                    298 non-null    int64  
 1   code                                       298 non-null    int64  
 2   Identity                                   298 non-null    object 
 3   E-mail                                     298 non-null    object 
 4   Name                                       298 non-null    object 
 5   Bank Name                                  298 non-null    object 
 6   e_institute                                298 non-null    object 
 7   e_contact                                  298 non-null    float64
 8   Branch Name                                298 non-null    object 
 9   Account No.                                298 non-null    float64
 10  IFSC Code                      

In [55]:
bill_rcvd["Level"].unique() 

array(['DIPLOMA', 'PG', 'UG'], dtype=object)

In [56]:
order = ['PG', 'UG', 'DIPLOMA']

bill_rcvd['Level'] = pd.Categorical(
    bill_rcvd['Level'],
    categories=order,
    ordered=True
)

/tmp/ipykernel_1375/2482446036.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bill_rcvd['Level'] = pd.Categorical(


In [57]:
bill_rcvd = bill_rcvd.sort_values(by=['E-mail', 'Level']).reset_index(drop=True)

In [58]:
bill_rcvd_final=bill_rcvd.iloc[:,[0,2,4,5,8,9,10,11,12,14,19,20]]
bill_rcvd_final.to_excel('/home/dharm/bill_rcvd_final.xlsx', index=False)

In [59]:
bill_pending = bill[bill['print'] != 'rcvd']
bill_pending["Level"].value_counts()

Level
UG    8
PG    4
Name: count, dtype: int64

# bill_rcvd

In [60]:
bill_rcvd['Level'].unique()

['UG', 'DIPLOMA', 'PG']
Categories (3, object): ['PG' < 'UG' < 'DIPLOMA']

In [61]:
bill_rcvd['pay'] = np.where(
    bill_rcvd['Level'] == 'UG',
    bill_rcvd['No. of Question PaperS Set'] * 750,
    bill_rcvd['No. of Question PaperS Set'] * 1000
)

In [62]:
path = '/tmp/pay.txt'
bill_rcvd[['pay']].to_csv(path, index=False, header=False)
print(f"File saved at: {path}")

File saved at: /tmp/pay.txt


# teacherwise payemnt

In [63]:
bill = pd.read_excel('../20feb.xlsx', sheet_name='bill')


bill_rcvd = bill[bill['print'] == 'rcvd']

In [64]:
bill_rcvd.info()

<class 'pandas.core.frame.DataFrame'>
Index: 298 entries, 0 to 297
Data columns (total 21 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Sr. No.                                    298 non-null    int64  
 1   code                                       298 non-null    int64  
 2   Identity                                   298 non-null    object 
 3   E-mail                                     298 non-null    object 
 4   Name                                       298 non-null    object 
 5   Bank Name                                  298 non-null    object 
 6   e_institute                                298 non-null    object 
 7   e_contact                                  298 non-null    float64
 8   Branch Name                                298 non-null    object 
 9   Account No.                                298 non-null    float64
 10  IFSC Code                      

In [65]:
bill_rcvd.groupby("E-mail")["Branch Name"].nunique().loc[lambda x: x > 1]

Series([], Name: Branch Name, dtype: int64)

In [66]:
teacher_wise_bill=bill_rcvd.groupby(["E-mail","Name","Bank Name", "Branch Name", "IFSC Code"])['Total Amount Payable after (5% deduction)'].sum()

In [67]:
teacher_wise_bill.to_excel('/home/dharm/teacher_wise_bill.xlsx')

In [68]:
bill_rcvd.groupby(["E-mail","Name","Bank Name","e_contact", "Branch Name", "Account No.","IFSC Code", "e_institute"])['Total Amount Payable after (5% deduction)'].sum()

E-mail                       Name                           Bank Name               e_contact     Branch Name                                Account No.   IFSC Code    e_institute                            
06001.bhawna@gmail.com       Dr. Bhawna Thakur              BANK OF INDIA           9.888135e+09  BANK OF INDIA MEHINDWANI BRANCH            6.547101e+14  BKID0002094  SGTB Khalsa College, Sri Anandpur Sahib    2850.0
5rajneeshthakur7@gmail.com   Dr. Rajinish Thakur            STATE BANK OF INDIA     8.219281e+09  FATEHGARH SAHIB                            3.869018e+10  SBIN0050591  -----                                      1140.0
academics.gnc24@gmail.com    Dr. Shaveta Dargan             HDFC SRI MUKTSAR SAHIB  8.146567e+09  HDFC, SRI MUKTSAR                          5.010019e+13  HDFC0000431  Guru Nanak College, Sri Muktsar Sahib      1995.0
                                                                                    8.360512e+09  HDFC, SRI MUKTSAR                       

In [69]:
result = bill_rcvd.groupby(
    ["Name", "E-mail","Bank Name","e_contact", "Branch Name", "Account No.","IFSC Code", "e_institute"],
    as_index=False
)['Total Amount Payable after (5% deduction)'].sum()

# Add Sr. No.
result.insert(0, "Sr. No.", range(1, len(result) + 1))

In [70]:
result.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 154 entries, 0 to 153
Data columns (total 10 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Sr. No.                                    154 non-null    int64  
 1   Name                                       154 non-null    object 
 2   E-mail                                     154 non-null    object 
 3   Bank Name                                  154 non-null    object 
 4   e_contact                                  154 non-null    float64
 5   Branch Name                                154 non-null    object 
 6   Account No.                                154 non-null    float64
 7   IFSC Code                                  154 non-null    object 
 8   e_institute                                154 non-null    object 
 9   Total Amount Payable after (5% deduction)  154 non-null    float64
dtypes: float64(3), int64(1), o

In [71]:
result.to_excel('teacher_wise_bill_with_sr_no.xlsx', index=False)